# Salary Factor Analysis and College Admissions PCA

**Recruiter-facing end-to-end analysis · Factorial inference and dimensionality reduction · Python 3.12/3.13**

> Education shows a large observed salary association (eta²=0.626); 6 standardized components retain at least 80% of college-indicator variance.

## Executive summary

**Objective:** Assess salary differences with robust sensitivity checks and compress correlated college indicators without hiding information loss.

**Data:** 40 salary observations and 777 colleges with 17 numeric admissions and institutional indicators.

**Verified result:** Education shows a large observed salary association (eta²=0.626); 6 standardized components retain at least 80% of college-indicator variance.

**Decision supported:** Identify material group differences and select a defensible reduced indicator set.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A compensation analyst or higher-education analyst.

**Decision:** Identify material group differences and select a defensible reduced indicator set.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '03-anova-pca-analysis'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 03-anova-pca-analysis


## 4. Data provenance and scope

College data corresponds to the ISLR College dataset; the small salary dataset's sampling process and license are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

                  file  size_mb           sha256
college_admissions.csv    0.073 d57bbc2f3edf6acb
       salary_data.csv    0.001 457fa1f5a40ee9de


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


college_admissions.csv: 18 columns
                       Names  Apps  Accept  Enroll  Top10perc  Top25perc  ...  PhD  Terminal  S.F.Ratio  perc.alumni  Expend  Grad.Rate
Abilene Christian University  1660    1232     721         23         52  ...   70        78       18.1           12    7041         60
          Adelphi University  2186    1924     512         16         29  ...   29        30       12.2           16   10527         56
              Adrian College  1428    1097     336         22         50  ...   53        66       12.9           30    8735         54
         Agnes Scott College   417     349     137         60         89  ...   92        97        7.7           37   19016         59
   Alaska Pacific University   193     146      55         16         44  ...   76        72       11.9            2   10922         15

salary_data.csv: 3 columns
 Education    Occupation  Salary
 Doctorate  Adm-clerical  153197
 Doctorate  Adm-clerical  115945
 Doctorate  Adm-cleri

## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 130 lines
Functions: _eta_squared, _cohen_d, _holm, _pairwise, _design_matrix, _nested_interaction, run_analysis


## 7. Methodology and hypotheses

Classic and Welch ANOVA, Kruskal sensitivity, Levene test, eta-squared, Holm-corrected pairwise Welch tests, interaction F-test, standardized PCA, loading interpretation, and reconstruction error.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_03_anova_pca_analysis", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 0.55 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


college_data_quality.csv (18 fields)
     column   dtype  missing_count  missing_percent  unique_values  constant
      Names  object              0              0.0            777     False
       Apps   int64              0              0.0            711     False
     Accept   int64              0              0.0            693     False
     Enroll   int64              0              0.0            581     False
  Top10perc   int64              0              0.0             82     False
  Top25perc   int64              0              0.0             89     False
F.Undergrad   int64              0              0.0            714     False
P.Undergrad   int64              0              0.0            566     False
   Outstate   int64              0              0.0            640     False
 Room.Board   int64              0              0.0            553     False
      Books   int64              0              0.0            122     False
   Personal   int64              0    

## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'salary_pairwise_holm.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'Education shows a large observed salary association (eta²=0.626); 6 standardized components retain at least 80% of college-indicator variance.')

Primary evidence: salary_pairwise_holm.csv, shape=(9, 7)
     field          group_1          group_2  mean_difference  cohen_d  welch_p_value  holm_adjusted_p_value
 Education        Bachelors        Doctorate      -43274.0667  -0.9658         0.0121                 0.0121
 Education        Bachelors          HS-grad       90114.1556   2.3077         0.0000                 0.0000
 Education        Doctorate          HS-grad      133388.2222   3.6348         0.0000                 0.0000
Occupation     Adm-clerical  Exec-managerial      -55693.3000  -1.2710         0.0106                 0.0637
Occupation     Adm-clerical   Prof-specialty      -27528.8538  -0.4116         0.3139                 0.9825
Occupation     Adm-clerical            Sales      -16180.1167  -0.2555         0.5448                 1.0000
Occupation  Exec-managerial   Prof-specialty       28164.4462   0.4181         0.2456                 0.9825
Occupation  Exec-managerial            Sales       39513.1833   0.6308 

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])

## 12. Visual evidence

### Anova Pca Evidence

![anova_pca_evidence](../reports/figures/anova_pca_evidence.png)

### Pca Explained Variance

![pca_explained_variance](../reports/figures/pca_explained_variance.png)

## 13. Business interpretation

Education shows a large observed salary association (eta²=0.626); 6 standardized components retain at least 80% of college-indicator variance.

The correct action is to use this result as evidence for **Identify material group differences and select a defensible reduced indicator set.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

The salary sample has only 40 observational records; robust alternatives and corrected pairwise tests reduce but do not eliminate small-sample and design limitations.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                   artifact  size_kb       sha256
     reports/figures/anova_pca_evidence.png    271.7 098e6583a7d7
 reports/figures/pca_explained_variance.png     58.4 9f3d87e2f1df
                       reports/metrics.json      6.2 9410e53e6394
reports/tables/college_component_scores.csv    109.5 430ff8f62361
    reports/tables/college_data_quality.csv      0.6 79e1e37e3ed5
            reports/tables/pca_loadings.csv      6.2 5e184862d8bc
     reports/tables/salary_data_quality.csv      0.2 1bd5075d2de0
    reports/tables/salary_pairwise_holm.csv      1.1 b695be7c0713


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed assess salary differences with robust sensitivity checks and compress correlated college indicators without hiding information loss. using classic and welch anova, kruskal sensitivity, levene test, eta-squared, holm-corrected pairwise welch tests, interaction f-test, standardized pca, loading interpretation, and reconstruction error. The final verified conclusion is: **Education shows a large observed salary association (eta²=0.626); 6 standardized components retain at least 80% of college-indicator variance.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/03-anova-pca-analysis/src/analysis.py
python scripts/execute_notebooks.py --project 03-anova-pca-analysis
```